<h1> Beamforming</h1>

In [ ]:
################# Change the values based on how much of the azimuth angles you want to see and the resolution ##################
# Define field of view in degrees that you want to process in theta, phi and range bins
def beamform_2d(beat_freq_data, phi_s, phi_e, phi_res, theta_s, theta_e, theta_res, x_locs, z_locs, r_idxs, radar_params, index):
    """
    Performs 2D beamforming along the azimuth (horizontal) dimension, this results in a bird eye view image.
    - beat_freq_data: beat data AKA the range FFT (size: num_x_stps * num_z_stps * num TX * num RX, num ADC samples)
    - phi_s: first azimuth angle that you want to start computing 
    - phi_e: last azimuth angle that you want to compute 
    - phi_res: resolution of the azimuth angles you want to compute
    - theta_s: first elevation angle that you want to start computing 
    - theta_e: last elevation angle that you want to compute 
    - theta_res: resolution of the elevation angles you want to compute
    - x_locs: x coordinate of antenna locations
    - z_locs: z coordinate of antenna locations
    - r_idx: range bins to calculate 
    - radar_parms: radar_params if needed 

    Returns:
    - sph_pwr: beamformed result (size: n_phi, n_theta, n_range)
    - phi: array of azimuth angles 
    - theta: array of elevation angles 
    """

    # Radar parameters
    sample_rate = radar_params["sample_rate"]
    num_samples = radar_params["num_samples"]
    slope = radar_params["slope"]
    lm = radar_params["lm"]
    num_z_stp = radar_params["num_z_stp"]
    num_tx = radar_params["num_tx"]
    num_rx = radar_params["num_rx"]
    adc_samples = radar_params["adc_samples"]

    # Convert angles to radians
    phi = np.arange(phi_s, phi_e, phi_res) * np.pi / 180 
    theta = np.arange(theta_s, theta_e, theta_res) * np.pi / 180
    num_theta = len(theta)
    num_phi = len(phi)

    # Initialize output
    sph_pwr = np.zeros((num_phi, num_theta, r_idxs.shape[0]), dtype=np.complex64)
    
    for i, angle_phi in enumerate(phi):
        for j, angle_theta in enumerate(theta):
            # Compute angles
            angles = x_locs * np.sin(angle_theta) * np.cos(angle_phi)
            # Compute phase shifts
            phase_shifts = np.exp((1j * 2 * np.pi / lm) * angles)
            # Apply phase shifts
            beamformed_signal = beat_freq_data[:, index:index+1,:] * phase_shifts[:,:, np.newaxis]
            # Compute sph
            sph_pwr[i, j, :] = np.abs(np.sum(np.sum(beamformed_signal, axis=0), axis=0))

    return sph_pwr, phi, theta

In [ ]:
#import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy
import scipy.io as sio
import time
#import open3d as o3d
import utils

Load data.

In [ ]:
# Define the path to the data
data_path = ('/Users/thomaskemper/Documents/EPFL/BA6/Communications project/Group_2/COM-304-Radars/project/data/input/data_mat/project/rdc_usejd_thomas_acote_separe.mat')

# Static data path
static_data_path = '/Users/thomaskemper/Documents/EPFL/BA6/Communications project/Group_2/COM-304-Radars/project/data/input/data_mat/project/rdc_vide.mat'

# Define the path to save the data
save_data_path = ('/Users/thomaskemper/Documents/EPFL/BA6/Communications project/Group_2/COM-304-Radars/project/data/output/project/beamforming_2d_output/rdc_usejd_thomas_acote_separe')

# loading data that is given
"""
    raw_data: is the raw radar data (after mixing) of size (num_x_stp x num_rx, num_z_stp, adc_samples)
    radar_params: is a dictionary with radar and position parameters: 'sample_rate', 'num_samples', 'slope', 'lm'(lambda), 'num_x_stp', 'num_z_stp', 'num_tx', 'num_rx', 'adc_samples'
"""

radar_params, raw_data = utils.load_raw_data(data_path)

Defining antenna positions. 

In [ ]:
x_ant_pos, z_pos, x_ant = utils.get_ant_pos_2d(radar_params['num_x_stp'], radar_params['num_z_stp'], radar_params['num_rx']) # this returns x position of rx, z positions of rx and tx, and x position of tx

Apply transform to the data.

In [ ]:
# Window hanning
window = np.hanning(radar_params['num_samples'])
adc_windowed = raw_data * window
range_fft = np.fft.fft(adc_windowed) # this is the range FFT of the data

Define the angles and range bins to run, and run the BF algo.

In [ ]:
# define the azimuth angles (horizontal FOV) that we want to look at 
r_idxs = np.arange(0, 140)
phi_s, phi_e = 0, 180
phi_res = 1
theta_s, theta_e = 70,110
theta_res = 1

# Run your algorithm here
bf_output1, phi, theta= beamform_2d(range_fft[:,:,r_idxs], phi_s, phi_e, phi_res, theta_s, theta_e, theta_res, x_ant_pos, z_pos, r_idxs, radar_params,0)

In [ ]:
# Ensure the directory exists
os.makedirs(save_data_path, exist_ok=True)

# Save the output
np.save(save_data_path + "/data_output.npy", bf_output1)

# Save the angles
np.savez(save_data_path +  "/data_angles.npz", phi= phi, theta= theta, r_idxs= r_idxs)

In [ ]:
# Load the output
load_data_path = '/Users/thomaskemper/Documents/EPFL/BA6/Communications project/Group_2/COM-304-Radars/project/data/output/project/beamforming_2d_output/rdc_usejd_thomas_acote_separe'

In [ ]:
# Load the output
bf_output1 = np.load(load_data_path + "/data_output" + ".npy")

# Load the angles
angles = np.load(load_data_path + "/data_angles" + ".npz")
phi = angles['phi']
theta = angles['theta']
r_idxs = angles['r_idxs']

In [ ]:
threshold = np.percentile(bf_output1, 96)
bf_output1 = np.where(bf_output1 > threshold, bf_output1, 0)

In [ ]:
from scipy.ndimage import median_filter

# Median filtering on magnitude:
bf_output1 = median_filter(np.abs(bf_output1), size=(1,1,1))

In [ ]:
# Plot the output from 2D Beamforming (You can change this as you see fit)
fig = plt.figure(figsize=(20, 20))

to_plot = np.sum(abs(bf_output1),axis=1)
to_plot = to_plot/np.max(np.reshape(to_plot,(1,-1)))
to_plot = to_plot**2
output_top = to_plot
ax1 = fig.add_subplot(142,projection = 'polar')
utils.plot_2d_heatmap(ax1, to_plot, phi, r_idxs, vmin=0, vmax=0.1)
ax1.title.set_text('Bird Eye View (Top View)')

plt.tight_layout()
plt.show()

In [ ]:
# Import
import numpy as np
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

In [ ]:
# --- Inputs ---
# heatmap: 2D array, shape (azimuth, z)
# phi_array: 1D array of azimuth angles in degrees (length = heatmap.shape[0])
# z_res: resolution along z-axis in meters
# radar_range: estimated mean range to convert angles into lateral meters

# remove close noise
output_top[:, 0:10] = 0

# Build full coordinate grid
phi_rad_2d, r_idxs_2d = np.meshgrid(phi, r_idxs, indexing='ij')  # shape: (180, 140)

x_coords_m = np.sin(phi_rad_2d) * r_idxs_2d  # shape: (180, 140)
z_coords_m = r_idxs_2d  # shape: (180, 140)

# Flatten for DBSCAN
points = np.stack([x_coords_m.ravel(), z_coords_m.ravel()], axis=1)
powers = output_top.ravel()

# Optional: keep only high-power points
threshold = np.percentile(powers, 96)
valid_mask = powers > threshold
points_thresh = points[valid_mask]

# --- DBSCAN ---
db = DBSCAN(eps=2.5, min_samples=90).fit(points_thresh)
labels = db.labels_
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print(f"Detected {n_clusters} clusters (people).")

In [ ]:
# Plot DBSCAN results
plt.figure(figsize=(6, 6))
output_top.T[:] = 0
plt.imshow(output_top.T, extent=[x_coords_m.min(), x_coords_m.max(), z_coords_m.min(), z_coords_m.max()],
           origin='lower', aspect='auto', cmap='hot')
plt.xlabel("X")
plt.ylabel("Y")
plt.title("DBSCAN Clustering on Full Heatmap")

# Plot clusters
for label in np.unique(labels):
    if label == -1:
        continue  # noise
    cluster_pts = points_thresh[labels == label]
    plt.scatter(cluster_pts[:,0], cluster_pts[:,1], s=30, label=f'Person {label+1}', alpha=0.7)

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot Front view
fig = plt.figure(figsize=(7, 7))

to_plot = np.sum(abs(bf_output1[:,:,20:40]),axis=-1)
to_plot = to_plot/np.max(np.reshape(to_plot,(1,-1)))
to_plot = to_plot**2
ax0 = fig.add_subplot()
utils.plot_2d_polar_heatmap(ax0, to_plot, phi, theta, vmin=0, vmax=0.1)
ax0.title.set_text('Front View')

plt.tight_layout()
plt.show()

In [ ]:
# Plot Side view
fig = plt.figure(figsize=(7, 7))

to_plot = np.sum(abs(bf_output1),axis=0)
to_plot = to_plot/np.max(np.reshape(to_plot,(1,-1)))
to_plot = to_plot**2
ax2 = fig.add_subplot()
utils.plot_2d_heatmap(ax2, to_plot, theta, r_idxs, vmin=0, vmax=0.1)
ax2.title.set_text('Side View')

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3d view
fig = plt.figure(figsize=(7, 7))

to_plot = bf_output1
to_plot = to_plot/np.max(np.reshape(to_plot,(1,-1)))
to_plot = to_plot**2
ax3 = fig.add_subplot(projection = '3d')
utils.plot_3d_polar_heatmap(ax3, to_plot, phi, theta, r_idxs, 0.1)
ax3.title.set_text('3d View')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
from tqdm import tqdm
from scipy.ndimage import median_filter
import time

# === File Paths ===
data_path = '/Users/thomaskemper/Documents/EPFL/BA6/Communications project/Group_2/COM-304-Radars/project/data/input/data_mat/project/rdc_usejd_marche.mat'
static_data_path = '/Users/thomaskemper/Documents/EPFL/BA6/Communications project/Group_2/COM-304-Radars/project/data/input/data_mat/project/rdc_vide_2.mat'

# === Load dynamic signal ===
radar_params, raw_data = utils.load_raw_data(data_path)  # (12, num_frames, 512)

# === FFT both ===
# Window hamming
window = np.hanning(radar_params['num_samples'])
adc_windowed = raw_data * window

# Apply FFT
range_fft = np.fft.fft(adc_windowed)

# === Antenna positions ===
x_ant_pos, z_ant_pos, x_ant = utils.get_ant_pos_2d(
    radar_params['num_x_stp'],
    radar_params['num_z_stp'],
    radar_params['num_rx']
)

# === Beamforming parameters ===
r_idxs = np.arange(0, 140)
phi_s, phi_e, phi_res = 0, 180, 1
theta_s, theta_e, theta_res = 70, 110, 1
phi = np.arange(phi_s, phi_e, phi_res) * np.pi / 180

theta = np.radians(np.arange(theta_s, theta_e, theta_res))
num_frames = radar_params['num_frames']

# === Precompute Beamforming ===
print("⚙️ Precomputing beamformed frames with noise removal...")
bf_frames = []
for i in tqdm(range(500)):
    #frame = beat_freq_super[:, i, r_idxs] # (12, 1, 512)
    bf, _, _ = beamform_2d(
        range_fft[:, :, r_idxs],
        phi_s, phi_e, phi_res,
        theta_s, theta_e, theta_res,
        x_ant_pos, z_ant_pos,
        r_idxs, radar_params, i
    )

    #threshold = np.percentile(bf, 98)
    #bf = np.where(bf > threshold, bf, 0)

    bf = median_filter(np.abs(bf), size=(1,1,1))

    bf = np.sum(abs(bf), axis=1)
    bf= bf/np.max(np.reshape(bf,(1,-1)))
    bf = bf ** 2
    bf_frames.append(bf)  # (phi, range)
# === Create animation ===
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='polar')

def update(frame_idx):
    ax.clear()
    to_plot = bf_frames[frame_idx]
    utils.plot_2d_heatmap(ax, to_plot, phi, r_idxs, vmin=0, vmax=0.1)
    ax.set_title(f'Top View Frame {frame_idx+1}/{num_frames}', va='bottom')
    return ax.collections

ani = animation.FuncAnimation(fig, update, frames=len(bf_frames), interval=50, blit=False)
ani.save("top_view_video_usejd_marche.gif", writer='pillow', fps=20)
plt.close(fig)
print("✅ Saved: top_view_video.gif")